# Compare a grid of robot designs

This notebook combines damping ratios and adhesive thicknesses, simulates each
design and ranks cases that pass physical screening. The leading cases are run
again at finer timesteps before selecting a finalist. You will inspect the grid,
read its comparison maps and explore individual trajectories.

Run the cells in order in JupyterLab with `.[analysis,notebooks]` installed.


## 1. Prepare the experiment

The provided grid contains 5 damping ratios × 5 adhesive thicknesses = 25 designs.
The recorded one-second study took about two minutes on CPU, before animation
rendering. Two original MATLAB cases give an estimated 32 minutes for 25 cases,
including per-case function generation; see [timing methodology](../docs/performance.md).


In [ ]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open from the repository or tutorials directory."
os.chdir(ROOT)
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/walking-robot-mpl")
sys.path.insert(0, str(ROOT / "tutorials"))
print("Python:", sys.executable)
from importlib.util import find_spec
from shlex import quote
required = ["jax", "yaml", "numpy", "matplotlib", "ipywidgets", "IPython", "walking_robot"]
missing = [name for name in required if find_spec(name) is None]
if missing:
    install = f"{quote(sys.executable)} -m pip install -e {quote(str(ROOT) + '[analysis,notebooks]')}"
    raise ModuleNotFoundError(
        "Missing packages in the selected notebook kernel: " + ", ".join(missing)
        + "\nRun this command in a terminal, then restart the kernel:\n" + install
    )
import jax
jax.config.update("jax_enable_x64", True)
import yaml
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image
from _widgets import animation_view, preview_design, trajectory_view, show_checks
print("Repository:", ROOT)
print("Simulation device:", jax.devices())


## 2. Select the grid

Select a few values for a smaller experiment, or keep all 25 combinations.
The diagram shows which combinations will be simulated. Both selection lists
must contain at least one value. Use Ctrl/Cmd to select several entries.

This example holds voltage at 200 V and recomputes drive frequency for each
design using the MATLAB rule (`drive_frequency_hz: null`). It starts with the
feet supported (`initial_clearance_um: 0`). The separate original-drop example
uses 501 µm; that changes the contact transient.


In [ ]:
from copy import deepcopy
source_sweep = yaml.safe_load(Path("examples/sweeps/supported_25_cases.yaml").read_text())
damping_values = widgets.SelectMultiple(options=source_sweep["parameters"]["damping_ratio"],
    value=tuple(source_sweep["parameters"]["damping_ratio"]), description="Damping")
thickness_values = widgets.SelectMultiple(options=source_sweep["parameters"]["adhesive_thickness_um"],
    value=tuple(source_sweep["parameters"]["adhesive_thickness_um"]), description="Adhesive (µm)", style={"description_width": "initial"})

def preview_grid(damping, thickness):
    print(f"{len(damping)} × {len(thickness)} = {len(damping) * len(thickness)} designs")
    fig, ax = plt.subplots(figsize=(7, 3))
    if damping and thickness:
        xx, yy = np.meshgrid(thickness, damping)
        ax.scatter(xx, yy)
        ax.set_xticks(thickness)
        ax.set_yticks(damping)
    ax.set(xlabel="Adhesive thickness (µm)", ylabel="Damping ratio", title="Selected simulation grid")
    ax.grid(alpha=.2)
    display(fig)
    plt.close(fig)

grid_preview = widgets.interactive_output(preview_grid, {"damping": damping_values, "thickness": thickness_values})
display(widgets.HBox([damping_values, thickness_values]))
display(grid_preview)


## 3. Save the experiment and execute the sweep

This cell saves the selected grid and a base-design YAML together, then starts
the solver. Each execution uses a new directory to keep experiments distinct.
Edit the saved YAML to add other values or parameters for later studies.

Screening uses 10 µs steps; up to three passing finalists are checked at 5 and
2.5 µs. Physical limits use every integration sample. Saved trajectories are
reduced to 100 µs intervals for plotting and exchange with MATLAB.


In [ ]:
from uuid import uuid4
if not damping_values.value or not thickness_values.value:
    raise ValueError("Select at least one value on each axis.")
OUTPUT = Path("outputs/tutorial_sweep") / uuid4().hex[:12]
OUTPUT.mkdir(parents=True)
settings = deepcopy(source_sweep)
settings["parameters"] = {"damping_ratio": list(damping_values.value),
                           "adhesive_thickness_um": list(thickness_values.value)}
settings["base_design"] = "base_design.yaml"
(OUTPUT / "base_design.yaml").write_text(Path("examples/designs/matlab_supported.yaml").read_text())
CONFIG = OUTPUT / "sweep.yaml"
CONFIG.write_text(yaml.safe_dump(settings, sort_keys=False))
print(CONFIG.read_text())
subprocess.run([sys.executable, "examples/run_sweep.py", "--config", str(CONFIG), "--output", str(OUTPUT / "results"), "--export-mat"], check=True)
RESULTS = OUTPUT / "results"
report = json.loads((RESULTS / "report.json").read_text())
print("Passing screening:", report["screening_passed"], "/", report["total_cases"])
print("Selected finalist:", report["best_id"], "· verified speed:", report["best_speed_m_s"], "m/s")
print("Saved run:", OUTPUT)


## 4. Read the speed and penetration maps

The left map shows mean COM speed over the final half of the simulation. A ×
marks a physical failure, which excludes the design from the eligible ranking.
The right map shows maximum foot penetration. Compare these quantities together.
The selected finalist has additional finer-timestep evidence; other cases may
have screening results only.


In [ ]:
import csv
from html import escape
from IPython.display import HTML

display(Image(filename=str(RESULTS / "sweep.png")))
with (RESULTS / "results.csv").open() as stream:
    rows = list(csv.DictReader(stream))
columns = ["case_id", "damping_ratio", "adhesive_thickness_um", "frequency_hz", "objective_m_s",
           "physical_passed", "refinement_status", "verified_objective_m_s", "selected_best"]
head = "".join(f"<th>{escape(name)}</th>" for name in columns)
body = "".join("<tr>" + "".join(f"<td>{escape(row[name])}</td>" for name in columns) + "</tr>" for row in rows)
display(HTML(f"<div style='overflow-x:auto'><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>"))


## 5. Explore an individual case

Choose a case to see its screening checks, design and motion. The time slider
shows the reduced saved trajectory. The reported checks retain full-resolution
peaks. This view uses screening trajectories for a consistent comparison across
all cases; finalist verification results are listed in the table above.


In [ ]:
case_options = [(f"Case {row['id']} · damping {row['design']['damping_ratio']:g} · adhesive {row['design']['adhesive_thickness_um']:g} µm", row["id"])
                for row in report["screening_cases"]]
case_selector = widgets.Dropdown(options=case_options, description="Design", layout=widgets.Layout(width="90%"))

def inspect_case(case_id):
    case = next(row for row in report["screening_cases"] if row["id"] == case_id)
    print(yaml.safe_dump(case["design"], sort_keys=False))
    show_checks(case["validation"])
    display(trajectory_view(RESULTS / "screen" / f"{case_id:03d}.npz",
                            foot_limit_m=case["validation"]["limits"]["max_foot_penetration_m"]))

case_output = widgets.interactive_output(inspect_case, {"case_id": case_selector})
display(case_selector, case_output)


## 6. Animate the verified finalist

When a finalist passes both physical and refinement checks, `best_design.yaml`
records it and `best_case.mat` contains its parameters and reduced trajectory.
The next cell renders the saved finest-resolution trajectory into an HTML movie.
If no finalist passes, inspect the failed cases before changing the experiment.


In [ ]:
if report["best_id"] is not None:
    from walking_robot.visualization import render_trajectory
    from walking_robot.validation import PhysicalLimits
    finest = min(settings["verification"]["timesteps_s"])
    filename = RESULTS / "verification" / f"dt_{finest:.9g}" / f"{report['best_id']:03d}.npz"
    with np.load(filename, allow_pickle=False) as saved:
        render_trajectory(saved["times"], saved["states"], saved["parameters"], RESULTS / "best",
                          PhysicalLimits(**report["screening_cases"][report["best_id"]]["validation"]["limits"]),
                          formats=["html"], max_frames=150)
    display(animation_view(RESULTS / "best" / "robot.html"))
else:
    print("No verified finalist in this experiment. Inspect physical failures and refinement status above.")


## 7. Compare the selected design with MATLAB

The exchange file exports data. To perform the separate short MATLAB check, set
`RUN_MATLAB = True` and provide the MATLAB executable below. This checks equations
and a 0.02-second matched trajectory; a full-duration MATLAB study is a separate
validation of the ranking. See [the sweep guide](../docs/sweeps.md).

To refine every passing design in a future experiment, set
`verification.top_cases` to the number of combinations in the saved sweep YAML.
The selected finalist is the best verified case among those checked.


In [ ]:
RUN_MATLAB = False
MATLAB_EXECUTABLE = "matlab"
if RUN_MATLAB and report["best_id"] is not None:
    subprocess.run([sys.executable, "tools/check_sweep_matlab.py", "--design", str(RESULTS / "best_design.yaml"),
                    "--matlab", MATLAB_EXECUTABLE, "--timestep", "0.00000125",
                    "--output", str(RESULTS / "matlab_check.json")], check=True)
else:
    print("The selected design and exchange data are saved under", RESULTS)
